#  Willy Wonka Chocolate Factory Dataset

## Project Goal

The goal of this project is to transform the Willy Wonka Chocolate Factory transactional dataset into a structured relational database in order to analyze sales performance, profitability, and operational efficiency through SQL and Python.

By combining data cleaning, database design, SQL analysis, and data visualization techniques, this project aims to identify:

- High-performing and low-performing products
- Regional sales and profitability trends
- Factory performance differences
- Business opportunities for optimization and growth

The final objective is to support data-driven decision-making within the Willy Wonka Chocolate Factory by delivering actionable insights and business recommendations.

## Hypotheses

**H1:** Premium chocolate bar products generate significantly higher profit margins than smaller candy products.

**H2:** Some regions consistently outperform others in overall sales and profitability.

**H3:** Certain factories contribute disproportionately to lower profitability levels.

# 1. Data Loading

In this section, we load the original transactional dataset and inspect its structure.  
Each row in the original dataset represents one sales transaction.

In [ ]:
import pandas as pd

# Load the original transactional dataset
df = pd.read_csv("wonka_choc_factory.csv")

# Preview the first rows to understand the structure of the dataset
df.head()

# 2. Initial Cleaning

Before creating the relational database structure, we standardize column names and convert date columns into the correct format.

In [ ]:
# Standardize column names for SQL compatibility:
# - remove extra spaces
# - convert to lowercase
# - replace spaces and slashes with underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
)

df.columns

In [ ]:
# Convert date columns into datetime format.
# This ensures that order and shipping dates can be correctly analyzed and loaded into SQL as DATE fields.
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

In [ ]:
# Remove unnecessary index-like column from the original CSV.
# This column does not contain business information and should not be included in the database.
df = df.drop(columns=["unnamed:_0"], errors="ignore")

# 3. Data Quality Checks

Before building the relational tables, we checked the dataset for missing values and duplicated rows.  
This step ensures that the database will be loaded with clean and reliable data.

In [ ]:
# Check missing values by column
df.isnull().sum()

In [ ]:
# Check duplicated rows
df.duplicated().sum()

In [ ]:
# Remove duplicated rows if any exist
df = df.drop_duplicates()

# 4. Data Examination & Schema Design

The goal of Day 2 is to transform the original flat CSV file into a structured relational database.  
Before designing the database, we analyzed each field to understand its business meaning and decide how it should be modeled.

Each row in the original dataset represents a sales transaction.  
The dataset includes product information, customer information, geographical information, sales metrics, profitability metrics, factory information, and order details.

Understanding the role of each column helps us decide which fields belong together and how to separate the dataset into relational tables.

# 5. Database Design Strategy

To improve scalability, reduce redundancy, and support efficient SQL analysis, the original flat dataset was transformed into a normalized relational database.

The database was designed following relational modeling principles:

- Separation of entities into independent tables
- Primary and foreign key relationships
- Reduction of duplicated information
- Improved query readability and maintainability

Normalization allows:

- Better data consistency
- Reduced storage redundancy
- Easier maintenance
- More efficient SQL JOIN operations
- Scalable analytical workflows

# 6. Relational Database Design Logic

The original dataset is a flat table where product, customer, location, factory, order, and sales information are stored together.

To create a normalized SQL database, we split the dataset into dimension and fact-like tables:

- `Customers`: unique customer identifiers.
- `Locations`: geographic information related to each transaction.
- `Factories`: factory names related to product production.
- `Products`: product-level information, connected to factories.
- `Orders`: order-level information, connected to customers and locations.
- `Order_Details`: transactional sales metrics, connected to orders and products.

This design reduces repetition, improves consistency, and supports SQL analysis through relational joins.

# 7. Customers Table

The `Customers` table stores unique customer identifiers.  
This table allows us to connect each order to a specific customer without repeating customer IDs unnecessarily in multiple places.

In [ ]:
# Create Customers table with unique customer IDs
customers = (
    df[["customer_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

customers.head()

In [ ]:
# Export Customers table for MySQL import
customers.to_csv("customers.csv", index=False, sep=",", encoding="utf-8-sig")

# 8. Locations Table

The `Locations` table stores geographic information related to each transaction.  
A unique `location_id` is created to connect locations with orders in the SQL database.

In [ ]:
# Create Locations table using geographic fields
locations = (
    df[[
        "country_region",
        "region",
        "state_province",
        "city",
        "postal_code",
        "latitude",
        "longitude"
    ]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create a unique primary key for each location
locations["location_id"] = locations.index + 1

# Reorder columns to place the primary key first
locations = locations[[
    "location_id",
    "country_region",
    "region",
    "state_province",
    "city",
    "postal_code",
    "latitude",
    "longitude"
]]

locations.head()

In [ ]:
# Export Locations table for MySQL import
locations.to_csv("locations.csv", index=False, sep=",", encoding="utf-8-sig")

# 9. Factories Table

The `Factories` table stores the list of factories associated with product production.  
A unique `factory_id` is created so that products can reference factories through a foreign key.

In [ ]:
# Create Factories table with unique factory names
factories = (
    df[["factory"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create a unique primary key for each factory
factories["factory_id"] = factories.index + 1

# Reorder columns
factories = factories[["factory_id", "factory"]]

# Rename column to match the SQL schema
factories = factories.rename(columns={"factory": "factory_name"})

factories.head()

In [ ]:
# Export Factories table for MySQL import
factories.to_csv("factories.csv", index=False, sep=",", encoding="utf-8-sig")

# 10. Products Table

The `Products` table stores product-level information such as product name, division, unit cost, and the factory where the product is produced.

We merge the product data with the `Factories` table to assign each product a `factory_id`, which will act as a foreign key in SQL.

In [ ]:
# Create Products table using product-related fields
products = (
    df[["product_name", "division", "cost", "factory"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Prepare factories lookup with original factory column name for merging
factory_lookup = factories.rename(columns={"factory_name": "factory"})

# Add factory_id to each product
products = products.merge(
    factory_lookup,
    on="factory",
    how="left"
)

# Create a unique primary key for each product
products["product_id"] = products.index + 1

# Select and reorder columns
products = products[[
    "product_id",
    "product_name",
    "division",
    "cost",
    "factory_id"
]]

# Rename cost to unit_cost to better describe product-level cost
products = products.rename(columns={"cost": "unit_cost"})

products.head()

In [ ]:
# Export Products table for MySQL import
products.to_csv("products.csv", index=False, sep=",", encoding="utf-8-sig")

# 11. Orders Table

The `Orders` table stores order-level information, including customer, location, order date, ship date, and ship mode.

Since the original dataset did not include a unique order identifier, we created an artificial `order_id` based on the dataframe index.  
This allows each transaction to be uniquely identified and connected to the `Order_Details` table.

In [ ]:
# Create an artificial order_id because the original dataset does not include a unique order identifier
df["order_id"] = df.index + 1

# Merge location_id into the main dataframe using geographic columns
orders = df.merge(
    locations,
    on=[
        "country_region",
        "region",
        "state_province",
        "city",
        "postal_code",
        "latitude",
        "longitude"
    ],
    how="left"
)

# Select order-level columns
orders = orders[[
    "order_id",
    "customer_id",
    "location_id",
    "order_date",
    "ship_date",
    "ship_mode"
]].drop_duplicates()

orders.head()

In [ ]:
# Export Orders table for MySQL import
orders.to_csv("orders.csv", index=False, sep=",", encoding="utf-8-sig")

# 12. Order_Details Table

The `Order_Details` table stores the transactional metrics for each order, including sales, units, gross profit, and cost.

This table connects each order to its corresponding product through `order_id` and `product_id`.  
It will be the main table used for SQL analysis of revenue, profit, and product performance.

In [ ]:
# Create a product lookup table that includes product_id and the original factory name
product_lookup = products.merge(
    factory_lookup,
    on="factory_id",
    how="left"
)

# Merge product_id into the original dataframe
order_details = df.merge(
    product_lookup,
    left_on=["product_name", "division", "cost", "factory"],
    right_on=["product_name", "division", "unit_cost", "factory"],
    how="left"
)

# Select transactional sales metrics
order_details = order_details[[
    "order_id",
    "product_id",
    "sales",
    "units",
    "gross_profit",
    "cost"
]]

# Create a unique primary key for each order detail row
order_details["order_detail_id"] = order_details.index + 1

# Reorder columns
order_details = order_details[[
    "order_detail_id",
    "order_id",
    "product_id",
    "sales",
    "units",
    "gross_profit",
    "cost"
]]

order_details.head()

In [ ]:
# Validate that no missing keys were created during the merge
order_details.isnull().sum()

In [ ]:
# Export Order_Details table for MySQL import
order_details.to_csv("order_details.csv", index=False, sep=",", encoding="utf-8-sig")

# 13. Final Validation

After generating all relational CSV files, we validate the shape of each table.  
These files are ready to be imported into MySQL following the database schema and ERD.

In [ ]:
print("Customers:", customers.shape)
print("Locations:", locations.shape)
print("Factories:", factories.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Details:", order_details.shape)

In [ ]:
# Validate uniqueness of primary keys
print("Customers PK valid:", customers["customer_id"].nunique() == len(customers))
print("Locations PK valid:", locations["location_id"].nunique() == len(locations))
print("Factories PK valid:", factories["factory_id"].nunique() == len(factories))
print("Products PK valid:", products["product_id"].nunique() == len(products))
print("Orders PK valid:", orders["order_id"].nunique() == len(orders))
print("Order Details PK valid:", order_details["order_detail_id"].nunique() == len(order_details))

In [ ]:
# Final check to confirm all tables were exported successfully
csv_files = [
    "customers.csv",
    "locations.csv",
    "factories.csv",
    "products.csv",
    "orders.csv",
    "order_details.csv"
]

csv_files

# 14. MySQL Import Order

The generated CSV files should be imported into MySQL in the following order:

1. Customers
2. Locations
3. Factories
4. Products
5. Orders
6. Order_Details

This order is important because some tables depend on others through foreign key relationships.

For example:

- `Products` depends on `Factories`
- `Orders` depends on `Customers` and `Locations`
- `Order_Details` depends on `Orders` and `Products`

Once imported into MySQL, the database can be validated using JOIN queries to confirm that the relational structure works correctly.

# 15. Next Step

After generating and validating the relational tables, the next phase of the project focuses on:

- Importing the data into MySQL
- Building SQL queries for business analysis
- Generating KPIs and profitability insights
- Creating visualizations to support decision-making